# S9 - Sistemas expertos (Pagarium)

**Objetivo (RA5-a/c):** entender la **anatomía** de un sistema experto y el ciclo *reconocer-resolver-actuar* construyendo un micro-motor, viendo **por qué falla** y comparándolo con `experta` y CLIPS.

> Hilo conductor: **Pagarium**, una pasarela de pagos que decide si aprueba o rechaza una transacción.

## El caso: Pagarium

Pagarium es una **pasarela de pagos** (como Stripe o Redsys). Cada pago trae unos **hechos** y, con reglas, Pagarium decide si **aprobar**, **revisar** o **rechazar**:

| Hecho | Ejemplo |
|---|---|
| `importe` | 200 |
| `antiguedad_meses` | 10 |
| `n_intentos` | 5 |

- importe pequeño y pocos reintentos → **aprobar**
- demasiados reintentos → **rechazar** (puede ser fraude)

### Python en un vistazo (lo justo para esta sesión)

| Lo que verás | Qué significa |
|---|---|
| `# texto` | **Comentario**: Python lo ignora; es una nota para ti |
| `nombre = valor` | **Guarda** un valor con un nombre (una *variable*) |
| `{ "clave": "valor", ... }` | Un **diccionario**: parejas nombre → valor |
| `[a, b, c]` | Una **lista**: varios valores ordenados |
| `def nombre(...):` | Crea una **función** (bloque de código con nombre) |
| `class Nombre:` | Crea una **clase** (plantilla de objetos) |
| `self` | Dentro de una clase, se refiere "a este objeto" |
| `lambda x: ...` | Una función corta escrita en una sola línea |
| `if ...:` / `for ... in ...:` / `while ...:` | Condición / repetición |
| `print(...)` | Escribe en pantalla |

**No necesitas saber programar de memoria**: lee los comentarios `#` y ejecuta las celdas de arriba a abajo.

## 1. Un micro-motor (lo construimos paso a paso)

Vamos a montarlo **pieza a pieza**, pero **en una sola celda**: en Python una clase no se puede repartir entre celdas (los métodos van indentados dentro de ella). Completa cada `# TODO` y ejecuta. Al final verás **por qué falla**.

### Los 4 pasos (dentro de la misma clase)

1. `__init__` — los tres almacenes.
2. `hecho` — añadir hechos.
3. `regla` — añadir reglas.
4. `run` — el ciclo reconocer-actuar.

> ⚠️ **Importante:** los `# TODO` son el ejercicio. Si ejecutas la celda **sin completarlos** (sin borrar el `pass`), el código dará error al usarlo (p. ej. `AttributeError: 'MicroMotor' object has no attribute 'traza'`). Es lo esperado: rellena los 4 métodos antes del Paso 5. La solución completa está en `sesion09_sistemas_expertos_soluciones.ipynb`.

In [10]:
class MicroMotor:
    # ---------- PASO 1 · los tres almacenes ----------
    def __init__(self):
        #  crea con self (el punto se lee "de este objeto"):
           self.hechos = {}   # diccionario para los hechos
           self.reglas = []   # lista para las reglas
           self.traza  = []   # lista para el orden de disparo
          #pass borra esta línea cuando completes el TODO

    # ---------- PASO 2 · añadir hechos ----------
    def hecho(self, **h):        # **h recoge las parejas nombre=valor
        #  añade esas parejas al diccionario self.hechos
               self.hechos.update(h)
        #pass

    # ---------- PASO 3 · añadir reglas ----------
    def regla(self, nombre, prioridad, condicion, accion):
        # añade (nombre, prioridad, condicion, accion) a self.reglas
              self.reglas.append((nombre, prioridad, condicion, accion))
        #pass

    # ---------- PASO 4 · el ciclo reconocer-actuar ----------
    def run(self):
        pendientes = True
        #  mientras pendientes sea True:

        while pendientes:
           pendientes = False
           for  nombre, prioridad, condicion, accion in self.reglas:
              if nombre in  self.traza
               continue

              if condicion(self.hechos):
                self.traza.append(nombre)
                accion(self.hechos)
                pendientes = True
        return self.hechos
        #   al salir, devuelve self.hechos
        #pass

### Paso 5 · Crear el motor y probarlo

In [11]:
motor = MicroMotor()
motor.hecho(importe=200, antiguedad_meses=10, n_intentos=5)   # hechos del pago

motor.regla("aprobar_pequena", 10, lambda h: h["importe"] < 500,      # condición: importe < 500
            lambda h: h.update(decision="aprobar"))                  # acción: decide "aprobar"
motor.regla("rechazar_reintentos", 10, lambda h: h["n_intentos"] >= 4,  # condición: reintentos >= 4
            lambda h: h.update(decision="rechazar"))                 # acción: decide "rechazar"

print(motor.run())     # -> {'importe': 200, ..., 'decision': 'rechazar'}  ¡y antes pasó por 'aprobar'!
print(motor.traza)     # -> ['aprobar_pequena', 'rechazar_reintentos']

{'importe': 200, 'antiguedad_meses': 10, 'n_intentos': 5, 'decision': 'rechazar'}
['aprobar_pequena', 'rechazar_reintentos']


El bug tiene **dos causas**: (1) no hay `resolve` (dispara todo lo que encaja) y (2) las reglas **no son excluyentes** (`aprobar` solo mira `importe`, no exige `n_intentos < 4`).

## 2. Actividad — corregir el conflicto con `salience`

Modifica `run()` para que, cuando varias reglas encajen, se dispare **solo la de mayor prioridad** (eso es `salience`). Comprueba que ya no hay contradicción.

> Pista: en vez de recorrer `for` todas las reglas, agrupa las que encajan y elige la de mayor `prioridad` con `max(...)`.

In [12]:
# =====================================================================
# ACTIVIDAD: corrige el conflicto del MicroMotor (resolución de conflictos).
# Haz DOS cambios sobre la clase del paso anterior:
#   (1) en run(): dispara SOLO la regla de mayor prioridad en cada vuelta.
#   (2) en la regla "aprobar": añade "and n_intentos < 4" (excluyente con "rechazar").
# =====================================================================

class MicroMotor2:
    def __init__(self):
        #crea los tres almacenes: self.hechos = {}, self.reglas = [], self.traza = []
        #pass
        self.hechos = {}
        self.reglas = []
        self.traza = []

    def hecho(self, **h):
        #  self.hechos.update(h)
        #pass
        self.hechos.update(h)

    def regla(self, nombre, prioridad, condicion, accion):
        #  self.reglas.append((nombre, prioridad, condicion, accion))
        #pass
        self.reglas.append((nombre, prioridad, condicion, accion))

    def run(self):
      # TODO (el cambio principal): dispara SOLO la de mayor prioridad.
        #   1) RECONOCER: lista de reglas activas
        #      [(p, n, a) for n, p, c, a in self.reglas if n not in self.traza and c(self.hechos)]
      while True:
        activas= [
            (prioridad, nombre, accion) for nombre, prioridad, condicion, accion
                                        in self.reglas
                                        if nombre not in self.traza
                                        and condicion(self.hechos)
        ]

        #   2) si no hay activas -> break
        if not activas:
            break

        #   3) RESOLVER: _, nombre, accion = max(activas, key=lambda t: t[0])
        _, nombre, accion = max(activas, key=lambda t: t[0])
        #   4) ACTUAR:
        self.traza.append(nombre)
        accion(self.hechos)
        #   5) al salir: return self.hechos
        return self.hechos


# Prueba (con n_intentos=5 debe acabar en 'rechazar'):
motor = MicroMotor2()
motor.hecho(importe=200, antiguedad_meses=10, n_intentos=5)

# TODO: regla "aprobar" -> importe < 500 AND n_intentos < 4 (prioridad 20)
motor.regla("aprobar_pequena", 20,
            lambda h: h["importe"] < 500 and h["n_intentos"] < 4, lambda h: h.update(decision="aprobar"))
# TODO: regla "rechazar" -> n_intentos >= 4 (prioridad 30)
motor.regla("rechazar_reintentos", 30,
            lambda h: h["n_intentos"] >= 4, lambda h: h.update(decision="rechazar"))

print(motor.run())     # esperado: decision = 'rechazar'
print(motor.traza)     # esperado: ['rechazar_reintentos']

{'importe': 200, 'antiguedad_meses': 10, 'n_intentos': 5, 'decision': 'rechazar'}
['rechazar_reintentos']


### Cómo funciona `run()` (línea a línea)

```python
def run(self):
    while True:                      # bucle: repite hasta que no quede nada que disparar
        # 1) RECONOCER (match): reglas activas
        activas = [(p, n, a) for n, p, c, a in self.reglas
                   if n not in self.traza and c(self.hechos)]
        if not activas:              # si no queda ninguna activa, termina
            break
        # 2) RESOLVER (resolve): elegir la de MAYOR prioridad
        _, nombre, accion = max(activas, key=lambda t: t[0])
        # 3) ACTUAR (act)
        self.traza.append(nombre)    # anota que ya se disparó
        accion(self.hechos)          # ejecuta su acción (modifica los hechos)
    return self.hechos
```

- `activas = [...]` es una **comprensión de lista**: recorre `self.reglas` y se queda con las que **no están en `self.traza`** y cuya **`condicion(self.hechos)` es `True`**. De cada una guarda la tupla `(prioridad, nombre, accion)` — la **prioridad va primera**, para poder ordenar por ella.
- `max(activas, key=lambda t: t[0])` elige la tupla con **mayor prioridad** (el `lambda` compara por el primer elemento). Con `_` descartamos la prioridad y nos quedamos con `nombre` y `accion`.
- `self.traza.append(nombre)` registra la regla para que **no se dispare dos veces**.
- `accion(self.hechos)` ejecuta la acción (`h.update(decision=...)`), que **cambia los hechos** y puede activar otras reglas.

### ¿Por qué `aprobar` = 20 y `rechazar` = 30?

La **prioridad (salience)** decide qué regla gana cuando varias encajan a la vez: **gana el número más alto**. En un sistema de pagos, ante la duda conviene **rechazar** (es lo seguro: demasiados reintentos = posible fraude); por eso `rechazar` tiene 30 y `aprobar` 20. Los números concretos son arbitrarios: lo importante es que **30 > 20**. En `experta` usamos los mismos valores para que se comporten igual.

**Matiz:** en este caso las dos reglas ya son **mutuamente excluyentes** (`aprobar` exige `n_intentos < 4` y `rechazar` `>= 4`), así que con `n_intentos = 5` solo encaja `rechazar` y la salience ni se usa. Es la **red de seguridad** por si dos reglas llegaran a encajar a la vez.

Con `importe=200` y `n_intentos=5`, solo encaja `rechazar_reintentos` (porque `aprobar` exige `n_intentos < 4`). Si probaras `n_intentos=2`, encajaría `aprobar`. La `salience` (30 > 20) es la red de seguridad cuando dos reglas encajan a la vez.

## 3. La misma política con `experta`

Experta es una librería de Python diseñada para la creación de **sistemas expertos** e inspirada en el lenguaje de programación **CLIPS**.

**🧠 ¿Qué es Experta?** Es una bifurcación (*fork*) de la antigua librería **Pyknow**. Su motor de inferencia funciona bajo el algoritmo **RETE** y permite definir una base de conocimiento mediante reglas y hechos (*Facts*) en Python estructurado.


Antes de escribir reglas, un minuto para los **decoradores**, porque `@DefFacts` y `@Rule` lo son.

### Los decoradores (`@`) en 2 minutos

Un **decorador** es una función que *envuelve* a otra para modificar su comportamiento, y se aplica con `@nombre`. Ejemplo (adaptado de [El Libro de Python](https://ellibrodepython.com/decoradores-python)):

In [13]:
def mi_decorador(funcion):        # recibe la función a decorar
    def nueva_funcion(a, b):      # la "envuelve"
        print("Antes de llamar")
        c = funcion(a, b)         # llama a la función original
        print("Después de llamar")
        return c
    return nueva_funcion          # devuelve la función envuelta

@mi_decorador                     # aplica el decorador a suma
def suma(a, b):
    return a + b

print(suma(5, 8))
# Antes de llamar
# 13
# Después de llamar

Antes de llamar
Después de llamar
13


Con **parámetros** (por eso `@Rule(...)` y `@DefFacts()` llevan paréntesis): el decorador se llama con argumentos y devuelve el decorador de verdad. En `experta`, `@Rule(...)` marca un método como **regla** y `@DefFacts()` marca el método de los **hechos iniciales**.

**Dos casos reales:**

- **Registrar en un fichero (`@log`).** Un decorador con parámetros que escribe el resultado de cada función en un fichero. Fíjate en las **tres funciones anidadas**.

In [14]:
def log(fichero_log):                     # decorador "de fábrica": recibe el nombre del fichero
    def decorador_log(func):              # el decorador de verdad: recibe la función
        def decorador_funcion(*args, **kwargs):       # *args/**kwargs = "cualquier argumento"
            with open(fichero_log, 'a') as f:         # abre el fichero en modo añadir
                output = func(*args, **kwargs)        # llama a la función original
                f.write(f"{output}\n")               # escribe el resultado en el fichero
        return decorador_funcion
    return decorador_log

@log('resultados.log')
def suma(a, b):
    return a + b

suma(10, 30)      # crea (o añade a) resultados.log la línea "40"

- **Permitir solo a usuarios autenticados.** Un decorador que ejecuta la función solo si la variable `autenticado` es `True`.

In [15]:
autenticado = True

def requiere_autenticacion(f):
    def funcion_decorada(*args, **kwargs):
        if not autenticado:                          # si no está autenticado
            print("Error. El usuario no se ha autenticado")
        else:
            return f(*args, **kwargs)                # si sí, ejecuta la función
    return funcion_decorada

@requiere_autenticacion
def di_hola():
    print("Hola")

di_hola()          # -> Hola (si autenticado=True) / Error... (si False)

Hola


### Lo construimos paso a paso (en una sola celda)

Igual que el micro-motor: la clase va **en una sola celda**. El parche y los `import` van al principio; completa cada `# TODO`.

In [18]:
%pip install experta

import collections, collections.abc
if not hasattr(collections, "Mapping"):   # parche para Python 3.10+
    collections.Mapping = collections.abc.Mapping
    collections.Iterable = collections.abc.Iterable
    collections.MutableMapping = collections.abc.MutableMapping

from experta import *                     # importa Fact, Rule, KnowledgeEngine...

class Pagarium(KnowledgeEngine):          # hereda el motor de reglas
    # ---------- PASO 2 · los hechos iniciales (@DefFacts) ----------
    @DefFacts()
    def inicio(self):
        # devuelve el hecho inicial accion="evaluar"
        #       (pista: yield Fact(accion="evaluar"))
        yield Fact(accion="evaluar")

    # ---------- PASO 3 · las reglas (@Rule) ----------
    @Rule(Fact(accion="evaluar"), Fact(importe=P(lambda i: i < 500)),
          Fact(n_intentos=P(lambda n: n < 4)), salience=20)   # salience = prioridad
    def aprobar(self):
        # declara decision="aprobar" con self.declare(Fact(...))
        self.declare(Fact(decision="aprobar"))

    @Rule(Fact(n_intentos=P(lambda n: n >= 4)), salience=30)   # más prioridad
    def rechazar(self):
        # declara decision="rechazar"
        self.declare(Fact(decision="rechazar"))

### Ejecutar el motor

In [22]:
motor = Pagarium()
motor.reset()      # crea el motor y lo deja listo
motor.declare(Fact(importe=200, n_intentos=5))   # hechos del pago
motor.declare(Fact(importe = 100 , n_intentos = 2))
motor.run()                            # ejecuta el ciclo de inferencia
print([dict(f) for f in motor.facts.values() if "decision" in f])   # solo hechos con 'decision'

[{'decision': 'rechazar', '__factid__': 4}, {'decision': 'aprobar', '__factid__': 5}]


### Cómo funciona la clase `Pagarium` (paso a paso)

```python
class Pagarium(KnowledgeEngine):
    @DefFacts()
    def inicio(self):
        yield Fact(accion="evaluar")

    @Rule(Fact(accion="evaluar"), Fact(importe=P(lambda i: i < 500)),
          Fact(n_intentos=P(lambda n: n < 4)), salience=20)
    def aprobar(self):
        self.declare(Fact(decision="aprobar"))

    @Rule(Fact(n_intentos=P(lambda n: n >= 4)), salience=30)
    def rechazar(self):
        self.declare(Fact(decision="rechazar"))
```

1. `class Pagarium(KnowledgeEngine):` — hereda del motor de `experta`, que **ya trae** el ciclo reconocer-resolver-actuar y la `salience`. Aquí no se programa el bucle: se programa el **conocimiento**.
2. `@DefFacts()` decora `inicio()`: es el método que devuelve los **hechos iniciales**, cargados al hacer `motor.reset()`.
3. `@Rule(...)` decora cada regla. La regla se dispara cuando se cumplen **todos** los `Fact(...)` de su decorador:
   - `aprobar` exige `accion="evaluar"`, `importe < 500` **y** `n_intentos < 4` (prioridad 20).
   - `rechazar` exige `n_intentos >= 4` (prioridad 30).
4. Dentro de cada regla, `self.declare(Fact(decision=...))` **añade un hecho nuevo** (la conclusión), que a su vez puede activar otras reglas.

Después se usa así:

```python
motor = Pagarium()
motor.reset()   # crea el motor y carga los hechos iniciales
motor.declare(Fact(importe=200, n_intentos=5))  # añade los hechos del pago
motor.run()                          # ejecuta el ciclo de inferencia
print([dict(f) for f in motor.facts.values() if "decision" in f])  # muestra las conclusiones
```

Con `n_intentos=5`, solo `rechazar` encaja (porque `aprobar` exige `n_intentos < 4`), así que se declara `decision="rechazar"`.

### ¿Por qué usamos `yield`?

En `@DefFacts()` escribimos `yield Fact(accion="evaluar")` en vez de `return`. `experta` espera que el método decorado sea un **generador**: un método que va «soltando» hechos con `yield`, uno por uno.

- `return` devolvería **una** cosa y terminaría el método.
- `yield` «pausa» el método y devuelve **un hecho cada vez**; si hubiera varios `yield`, iría devolviéndolos de uno en uno.

Aquí solo hay un hecho inicial (`accion="evaluar"`), pero `@DefFacts` exige ese estilo para poder cargar **varios** hechos iniciales (uno por cada `yield`).

### ¿Para qué sirven las `lambda` en este código?

Dentro de `@Rule(...)`, las condiciones se escriben con `P(lambda ...)`:

```python
Fact(importe=P(lambda i: i < 500))
```

- `P(...)` (de *predicado*) indica que la condición **no es un valor fijo**, sino una **comprobación** que se evalúa contra el valor del hecho.
- La `lambda` es una **función corta en una línea**: recibe un valor (`i` o `n`) y devuelve `True`/`False` (`i < 500`, `n < 4`).

Así, `Fact(importe=P(lambda i: i < 500))` significa «el hecho `importe` debe cumplir que su valor sea **menor que 500**». Compáralo con `Fact(importe=200)`, que exigiría el valor **exacto** 200.

**Ojo:** en `experta` las condiciones van **siempre** dentro de `P(...)`. Un `lambda` suelto (`Fact(importe=lambda i: i < 500)`) **no funciona**, porque `experta` compara por igualdad.

### Los tres ejemplos de RETE con `experta`

En los apuntes (sección 9) has visto el algoritmo RETE y cómo `experta`, CLIPS o Drools lo usan por debajo. Aquí tienes los **tres ejemplos** de la pasarela Pagarium listos para ejecutar. Observa la salida y compárala con lo explicado en los apuntes.

#### Ejemplo 1 · Detección de fraude en tiempo real

Dos reglas de fraude comparten la condición `monto > 5000`. RETE la evalúa **una sola vez**: si el monto es menor, corta el camino para ambas reglas de inmediato.

In [23]:
import collections, collections.abc
if not hasattr(collections, "Mapping"):     # parche para Python 3.10+
    collections.Mapping = collections.abc.Mapping
    collections.Iterable = collections.abc.Iterable
    collections.MutableMapping = collections.abc.MutableMapping

from experta import KnowledgeEngine, Fact, Rule, MATCH, TEST

# 1. Definimos la estructura de una Transacción
class Transaccion(Fact):
    """Información que viaja a través de la pasarela Pagarium"""
    pass

# 2. Definimos el Motor de Inferencia (La Red RETE)
class PagariumFraudEngine(KnowledgeEngine):

    # Regla Alerta Alfa: Monto > 5000 y los países no coinciden
    @Rule(
        Transaccion(
            monto=MATCH.monto,
            pais_tarjeta=MATCH.p_tarjeta,
            pais_ip=MATCH.p_ip
        ),
        # RETE evalúa el monto primero y comparte este filtro con otras reglas
        TEST(lambda monto: monto > 5000),
        TEST(lambda p_tarjeta, p_ip: p_tarjeta != p_ip)
    )
    def alerta_alfa(self):
        print("🚨 [ALERTA ALFA] -> Requiere Verificación Manual: Monto alto y discrepancia de países.")

    # Regla Alerta Beta: Monto > 5000 y comercio nuevo
    @Rule(
        Transaccion(
            monto=MATCH.monto,
            antiguedad_comercio_dias=MATCH.antiguedad
        ),
        # RETE reutiliza la lógica si el monto ya fue evaluado
        TEST(lambda monto: monto > 5000),
        TEST(lambda antiguedad: antiguedad < 30)
    )
    def alerta_beta(self):
        print("⛔ [ALERTA BETA] -> Bloqueo Temporal: Monto alto en un comercio con menos de 30 días.")


# 3. Pruebas del sistema en tiempo real
if __name__ == "__main__":
    # Inicializamos el motor de Pagarium
    engine = PagariumFraudEngine()
    engine.reset()

    print("--- Procesando Transacción 1 (Normal) ---")
    # Esta transacción se descarta de inmediato por el primer nodo RETE (monto bajo)
    engine.declare(Transaccion(monto=100, pais_tarjeta="ES", pais_ip="ES", antiguedad_comercio_dias=120))
    engine.run()

    print("\n--- Procesando Transacción 2 (Sospechosa Alfa) ---")
    # Monto alto y países distintos
    engine.reset()
    engine.declare(Transaccion(monto=6500, pais_tarjeta="US", pais_ip="ES", antiguedad_comercio_dias=90))
    engine.run()

    print("\n--- Procesando Transacción 3 (Sospechosa Beta) ---")
    # Monto alto y comercio muy nuevo
    engine.reset()
    engine.declare(Transaccion(monto=7000, pais_tarjeta="MX", pais_ip="MX", antiguedad_comercio_dias=15))
    engine.run()


--- Procesando Transacción 1 (Normal) ---

--- Procesando Transacción 2 (Sospechosa Alfa) ---
🚨 [ALERTA ALFA] -> Requiere Verificación Manual: Monto alto y discrepancia de países.

--- Procesando Transacción 3 (Sospechosa Beta) ---
⛔ [ALERTA BETA] -> Bloqueo Temporal: Monto alto en un comercio con menos de 30 días.


**Puntos clave:** `Fact` es el contenedor de datos que alimenta la red; `MATCH` y `TEST` enlazan propiedades y aplican condiciones; `engine.run()` activa el motor y la transacción se «filtra» por los nodos que corresponden.

**Qué sale y por qué:** la transacción de 100 € se descarta en el primer nodo (monto bajo). La de 6500 € con países distintos dispara **ALERTA ALFA**; la de 7000 € con comercio nuevo dispara **ALERTA BETA**.

#### Ejemplo 2 · Enrutamiento y comisiones

La red **indexa los caminos**: si llega un cliente `Premium`, se ignora toda la rama `Estándar`. Tres reglas eligen proveedor y comisión según el tipo de cliente y el método de pago.

In [ ]:
import collections, collections.abc
if not hasattr(collections, "Mapping"):     # parche para Python 3.10+
    collections.Mapping = collections.abc.Mapping
    collections.Iterable = collections.abc.Iterable
    collections.MutableMapping = collections.abc.MutableMapping

from experta import KnowledgeEngine, Fact, Rule, MATCH

# 1. Definimos la estructura de la transacción
class Pago(Fact):
    """Información del pago a procesar en Pagarium"""
    pass

# 2. Motor de Inferencia para Enrutamiento y Comisiones
class PagariumRoutingEngine(KnowledgeEngine):

    # Regla 1: Cliente Premium - Método Visa
    @Rule(Pago(tipo_cliente="Premium", metodo_pago="Visa", monto=MATCH.monto))
    def premium_visa(self, monto):
        comision = monto * 0.01  # 1%
        print(f"💳 [PREMIUM-VISA] -> Procesando por Redsys. Comisión (1%): ${comision:.2f}")

    # Regla 2: Cliente Premium - Método Cripto
    @Rule(Pago(tipo_cliente="Premium", metodo_pago="Cripto", monto=MATCH.monto))
    def premium_crypto(self, monto):
        comision = monto * 0.005  # 0.5%
        print(f"🪙 [PREMIUM-CRYPTO] -> Procesando por Blockchain. Comisión (0.5%): ${comision:.2f}")

    # Regla 3: Cliente Estándar - Método Visa
    @Rule(Pago(tipo_cliente="Estandar", metodo_pago="Visa", monto=MATCH.monto))
    def estandar_visa(self, monto):
        comision = monto * 0.025  # 2.5%
        print(f"💳 [ESTANDAR-VISA] -> Procesando por Redsys. Comisión (2.5%): ${comision:.2f}")


# 3. Simulación de la Pasarela de Pagos
if __name__ == "__main__":
    engine = PagariumRoutingEngine()

    print("--- Transacción A: Cliente VIP usando Tarjeta ---")
    engine.reset()
    engine.declare(Pago(tipo_cliente="Premium", metodo_pago="Visa", monto=1000))
    engine.run()
    # RETE entra por la rama Premium -> Visa y ejecuta la Regla 1.

    print("\n--- Transacción B: Cliente VIP usando Bitcoin ---")
    engine.reset()
    engine.declare(Pago(tipo_cliente="Premium", metodo_pago="Cripto", monto=2000))
    engine.run()
    # RETE entra por la rama Premium -> Cripto y ejecuta la Regla 2.

    print("\n--- Transacción C: Comercio común usando Tarjeta ---")
    engine.reset()
    engine.declare(Pago(tipo_cliente="Estandar", metodo_pago="Visa", monto=500))
    engine.run()
    # RETE ignora toda la rama Premium desde el primer nodo y se va directo a Estándar -> Visa (Regla 3).


**Qué sale y por qué:** Premium+Visa → 1 % (Redsys); Premium+Cripto → 0.5 % (Blockchain); Estándar+Visa → 2.5 % (Redsys). La rama Premium ignora por completo la rama Estándar.

#### Ejemplo 3 · Motor unificado (fraude + comisiones)

Se combinan ambos motores con `salience`: primero se evalúa el fraude (prioridad 10) y, si la transacción sigue «pendiente», se calcula la comisión (prioridad 1). La novedad es `AS.t << ...`, que vincula el hecho completo para poder modificarlo:

- `AS.t << TransaccionCompleta(...)` guarda el objeto `Fact` completo en `t`.
- `self.modify(t, status="bloqueado_...")` modifica el hecho original dentro de la red RETE.
- `MATCH.propiedad` extrae **un valor** individual; `TEST(lambda ...)` aplica **una condición**.

In [24]:
import collections, collections.abc
if not hasattr(collections, "Mapping"):     # parche para Python 3.10+
    collections.Mapping = collections.abc.Mapping
    collections.Iterable = collections.abc.Iterable
    collections.MutableMapping = collections.abc.MutableMapping

from experta import KnowledgeEngine, Fact, Rule, MATCH, TEST, AS

# 1. Estructura de la transacción con estado de validación
class TransaccionCompleta(Fact):
    """Información total de la transacción que procesa Pagarium"""
    pass

# 2. Motor de Inferencia Unificado
class PagariumUnifiedEngine(KnowledgeEngine):

    # ==========================================
    # BLOQUE 1: REGLAS DE FRAUDE (Prioridad Alta: salience=10)
    # ==========================================

    @Rule( # t de transaccion
        AS.t << TransaccionCompleta( # <--- Recibe la variable 't' aquí
            monto=MATCH.monto,
            pais_tarjeta=MATCH.p_tarjeta,
            pais_ip=MATCH.p_ip,
            status="pendiente" # Solo evaluar si está pendiente
        ),
        TEST(lambda monto: monto > 5000),
        TEST(lambda p_tarjeta, p_ip: p_tarjeta != p_ip),
        salience=10
    )
    def detectar_fraude_alfa(self, t):
        print("🚨 [FRAUDE - ALFA] -> Discrepancia de países en monto alto.")
        # Modificamos el hecho en la red RETE para bloquearlo
        self.modify(t, status="bloqueado_manual") # <--- Modifica el hecho original usando 't'

    @Rule(
        AS.t << TransaccionCompleta(
            monto=MATCH.monto,
            antiguedad_comercio_dias=MATCH.antiguedad,
            status="pendiente"
        ),
        TEST(lambda monto: monto > 5000),
        TEST(lambda antiguedad: antiguedad < 30),
        salience=10
    )
    def detectar_fraude_beta(self, t):
        print("⛔ [FRAUDE - BETA] -> Comercio demasiado nuevo para este monto.")
        self.modify(t, status="bloqueado_temporal")


    # ==========================================
    # BLOQUE 2: REGLAS DE ENRUTAMIENTO (Prioridad Baja: salience=1)
    # Solo se ejecutan si status sigue siendo "pendiente"
    # ==========================================

    @Rule(
        TransaccionCompleta(tipo_cliente="Premium", metodo_pago="Visa", monto=MATCH.monto, status="pendiente"),
        salience=1
    )
    def enrutar_premium_visa(self, monto):
        print(f"💳 [PROCESADO] -> Premium + Visa. Ruta: Redsys. Comisión: ${monto * 0.01:.2f}")

    @Rule(
        TransaccionCompleta(tipo_cliente="Premium", metodo_pago="Cripto", monto=MATCH.monto, status="pendiente"),
        salience=1
    )
    def enrutar_premium_crypto(self, monto):
        print(f"🪙 [PROCESADO] -> Premium + Cripto. Ruta: Blockchain. Comisión: ${monto * 0.005:.2f}")

    @Rule(
        TransaccionCompleta(tipo_cliente="Estandar", metodo_pago="Visa", monto=MATCH.monto, status="pendiente"),
        salience=1
    )
    def enrutar_estandar_visa(self, monto):
        print(f"💳 [PROCESADO] -> Estándar + Visa. Ruta: Redsys. Comisión: ${monto * 0.025:.2f}")


# 3. Pruebas del Ciclo de Vida de Pagarium
if __name__ == "__main__":
    engine = PagariumUnifiedEngine()

    print("--- CASO 1: Transacción Limpia (Premium usando Visa) ---")
    engine.reset()
    engine.declare(TransaccionCompleta(
        monto=1200, pais_tarjeta="ES", pais_ip="ES",
        antiguedad_comercio_dias=150, tipo_cliente="Premium",
        metodo_pago="Visa", status="pendiente"
    ))
    engine.run()
    # RETE ve que no aplica fraude, pasa al bloque de enrutamiento.

    print("\n--- CASO 2: Transacción Fraudulenta (Intento de Fraude Beta) ---")
    engine.reset()
    engine.declare(TransaccionCompleta(
        monto=7500, pais_tarjeta="MX", pais_ip="MX",
        antiguedad_comercio_dias=10, tipo_cliente="Estandar",
        metodo_pago="Visa", status="pendiente"
    ))
    engine.run()
    # RETE ejecuta primero la regla de fraude por su salience. Al modificar el status
    # a "bloqueado_temporal", las reglas de enrutamiento ya no se activan.


--- CASO 1: Transacción Limpia (Premium usando Visa) ---
💳 [PROCESADO] -> Premium + Visa. Ruta: Redsys. Comisión: $12.00

--- CASO 2: Transacción Fraudulenta (Intento de Fraude Beta) ---
⛔ [FRAUDE - BETA] -> Comercio demasiado nuevo para este monto.


**Qué sale y por qué:** el caso 1 (limpio) pasa al enrutamiento y se procesa; el caso 2 (fraudulento) dispara **FRAUDE - BETA** y, al cambiar el `status` con `self.modify`, las reglas de enrutamiento ya no se activan.

## 4. La misma política en CLIPS (`clipspy`)

`clipspy` es el enlace de Python con **CLIPS 6.4** (producción). Las reglas se escriben en una sintaxis LISP con paréntesis, y la prioridad se marca con `(declare (salience 30))`.

In [25]:
%pip install clipspy
import clips

env = clips.Environment()              # crea el entorno de CLIPS
# build(...) define UNA regla cada vez (no dos a la vez). Cada llamada añade una regla.
env.build("(defrule aprobar (importe ?i&:(< ?i 500)) (n_intentos ?n&:(< ?n 4)) => (assert (decision aprobar)))")
env.build("(defrule rechazar (declare (salience 30)) (n_intentos ?n&:(>= ?n 4)) => (assert (decision rechazar)))")
#   (defrule nombre   condiciones-con-patrones       =>   acciones(assert añade hechos))

env.assert_string("(importe 200)")     # añade un hecho: importe 200
env.assert_string("(n_intentos 5)")    # añade otro hecho
env.run()                              # ejecuta
print([str(f) for f in env.facts()])   # lista todos los hechos finales

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 901.2/901.2 kB 10.8 MB/s eta 0:00:00
['(importe 200)', '(n_intentos 5)', '(decision rechazar)']


### Otro dominio: control de calefacción

El **mismo** micro-motor sirve para cualquier dominio; solo cambian las reglas y los hechos. Aquí controla una calefacción.

In [29]:
# El mismo micro-motor, otro dominio: calefacción
motor = MicroMotor()
motor.hecho(temperatura=18, hora=23, ventana_abierta=False)   # hechos del ambiente

motor.regla("frio", 10, lambda h: h["temperatura"] < 20, lambda h: h.update(encender=True))   # si hace frío, encender
motor.regla("noche", 10, lambda h: h["hora"] >= 22, lambda h: h.update(modo="eco"))           # si es de noche, modo eco
motor.regla("ventilar", 5, lambda h: h["ventana_abierta"], lambda h: h.update(encender=False))  # ventana abierta: no encender

print(motor.run())    # hechos finales
print(motor.traza)    # reglas disparadas

{'temperatura': 18, 'hora': 23, 'ventana_abierta': False, 'encender': True, 'modo': 'eco'}
['frio', 'noche']


### Otro dominio: clasificación de animales

`experta` con reglas de un zoólogo. Fíjate en el **encadenamiento hacia delante**: un hecho (`pelo`) dispara una regla que crea otro hecho (`mamifero`), y ese dispara la siguiente.

In [28]:
class Animales(KnowledgeEngine):
    @DefFacts()
    def inicio(self):
        yield Fact(analizar=True)      # arranque

    @Rule(Fact(analizar=True), salience=10)
    def datos(self):                    # carga los datos del animal a clasificar
        self.declare(Fact(pelo=True), Fact(carnivoro=True), Fact(color="leonado"), Fact(manchas="oscuras"))

    @Rule(Fact(pelo=True))              # si tiene pelo -> es mamífero (regla intermedia)
    def mamifero(self):
        self.declare(Fact(mamifero=True))

    @Rule(Fact(mamifero=True), Fact(carnivoro=True), Fact(color="leonado"), Fact(manchas="oscuras"))
    def guepardo(self):                 # todas las pistas encajan -> conclusión
        print("IDENTIFICADO: guepardo")

engine = Animales()
engine.reset()
engine.run()   # ejecuta: se ve el encadenamiento

IDENTIFICADO: guepardo


### Más reglas en CLIPS

Dos reglas que deciden la tarifa según el importe: a partir de 100 €, tarifa premium.

In [27]:
env = clips.Environment()
# (?i&:(< ?i 100))  significa: la variable ?i debe cumplir (< ?i 100) = "?i < 100"
env.build("(defrule pequena (importe ?i&:(< ?i 100)) => (assert (tarifa basica)))")
env.build("(defrule grande (importe ?i&:(>= ?i 100)) => (assert (tarifa premium)))")
env.assert_string("(importe 150)")   # importe 150 -> cumple la regla 'grande'
env.run()
print([str(f) for f in env.facts()])

['(importe 150)', '(tarifa premium)']


**Cierre · la resolución de conflictos (entrega).**

A lo largo de la sesión has resuelto **el mismo problema** —aprobar o rechazar un pago de Pagarium según `importe` y `n_intentos`— con **tres motores**: el micro-motor en Python, `experta` y CLIPS.

Haz estos tres pasos:

1. **Recupera la traza de cada motor** con los mismos hechos (`importe=200`, `n_intentos=5`). La *traza* es la secuencia de reglas que se disparan, en orden:
   - micro-motor → `print(motor.traza)`;
   - `experta` → el `print` que hace cada regla al dispararse;
   - CLIPS → los hechos que quedan al final (`print([str(f) for f in env.facts()])`).

2. **Compáralas.** Busca qué motor dispara **dos reglas contradictorias** (`aprobar` **y** `rechazar` a la vez) y cuáles disparan **solo una**.

3. **Escribe un párrafo de 3 líneas** explicando qué papel juega la **resolución de conflictos**: qué pasa cuando varias reglas encajan a la vez y cómo el motor elige cuál disparar (pista: la **prioridad**/`salience` y las reglas **mutuamente excluyentes**).

In [32]:
1.
# MISMO PAGO EN LOS TRES MOTORES:
# importe = 200, n_intentos = 5

# ---------- MICRO-MOTOR ORIGINAL ----------
micro = MicroMotor()
micro.hecho(importe=200, antiguedad_meses=10, n_intentos=5)

micro.regla(
    "aprobar_pequena", 10,
    lambda h: h["importe"] < 500,
    lambda h: h.update(decision="aprobar")
)

micro.regla(
    "rechazar_reintentos", 10,
    lambda h: h["n_intentos"] >= 4,
    lambda h: h.update(decision="rechazar")
)

micro.run()
print("MICRO-MOTOR")
print("Traza:", micro.traza)
print("Decisión final:", micro.hechos["decision"])


# ---------- EXPERTA ----------
# Añadimos mensajes para ver qué reglas se ejecutan.
class PagariumEntrega(KnowledgeEngine):
    @DefFacts()
    def inicio(self):
        yield Fact(accion="evaluar")

    @Rule(
        Fact(accion="evaluar"),
        Fact(importe=P(lambda i: i < 500)),
        Fact(n_intentos=P(lambda n: n < 4)),
        salience=20
    )
    def aprobar(self):
        print("Regla disparada: aprobar")
        self.declare(Fact(decision="aprobar"))

    @Rule(Fact(n_intentos=P(lambda n: n >= 4)), salience=30)
    def rechazar(self):
        print("Regla disparada: rechazar")
        self.declare(Fact(decision="rechazar"))

print("\nEXPERTA")
experto = PagariumEntrega()
experto.reset()
experto.declare(Fact(importe=200, n_intentos=5))
experto.run()

print("Decisiones:", [
    f["decision"]
    for f in experto.facts.values()
    if "decision" in f
])


# ---------- CLIPS ----------
print("\nCLIPS")
entorno = clips.Environment()

entorno.build("""
(defrule aprobar
    (importe ?i&:(< ?i 500))
    (n_intentos ?n&:(< ?n 4))
    =>
    (assert (decision aprobar)))
""")

entorno.build("""
(defrule rechazar
    (declare (salience 30))
    (n_intentos ?n&:(>= ?n 4))
    =>
    (assert (decision rechazar)))
""")

entorno.assert_string("(importe 200)")
entorno.assert_string("(n_intentos 5)")
entorno.run()

print("Hechos finales:", [str(f) for f in entorno.facts()])


MICRO-MOTOR
Traza: ['aprobar_pequena', 'rechazar_reintentos']
Decisión final: rechazar

EXPERTA
Regla disparada: rechazar
Decisiones: ['rechazar']

CLIPS
Hechos finales: ['(importe 200)', '(n_intentos 5)', '(decision rechazar)']


In [ ]:
print("""El micro-motor original dispara dos reglas contradictorias: primero aprobar y después rechazar. 
 Esto ocurre porque la regla de aprobación solo comprueba que el importe sea menor de 500 €, mientras que la de rechazo comprueba que haya al menos 4 intentos. 
 Aunque la decisión final es rechazar, ambas reglas se han ejecutado.
Experta y CLIPS solo disparan la regla de rechazo. Sus reglas de aprobación también exigen menos de 4 intentos, condición que este pago no cumple. 
El micro-motor corregido, MicroMotor2, también debería disparar únicamente la regla de rechazo.""")


El micro-motor original dispara dos reglas contradictorias: primero aprobar y después rechazar. Esto ocurre porque la regla de aprobación solo comprueba que el importe sea menor de 500 €, mientras que la de rechazo comprueba que haya al menos 4 intentos. Aunque la decisión final es rechazar, ambas reglas se han ejecutado. Experta y CLIPS solo disparan la regla de rechazo. Sus reglas de aprobación también exigen menos de 4 intentos, condición que este pago no cumple. El micro-motor corregido, MicroMotor2, también debería disparar únicamente la regla de rechazo.


In [ ]:
print("""La resolución de conflictos permite elegir qué regla ejecutar cuando varias cumplen sus condiciones al mismo tiempo.
La prioridad o salience hace que el motor ejecute primero la regla con el valor más alto, pero no impide por sí sola que después se ejecuten otras.
Las reglas mutuamente excluyentes evitan decisiones contradictorias, como aprobar y rechazar un mismo pago.""")

La resolución de conflictos permite elegir qué regla ejecutar cuando varias cumplen sus condiciones al mismo tiempo.  
La prioridad o salience hace que el motor ejecute primero la regla con el valor más alto, pero no impide por sí sola que después se ejecuten otras.  
Las reglas mutuamente excluyentes evitan decisiones contradictorias, como aprobar y rechazar un mismo pago.
